# Stack Overflow Data Collection
Collects questions via the API and cleans HTML bodies with BeautifulSoup. Saves to CSV and resumes if re-run.

In [1]:
!pip install -q requests beautifulsoup4 lxml pandas

## Import necessary dependencies

In [2]:
import csv, os, time
from datetime import datetime, timedelta, timezone
import requests
from bs4 import BeautifulSoup
import pandas as pd

/Users/bipin/Documents/aberdeen msc ai/applied_artificial_intelligence/assessment_1/assessment_1_task_1/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Constants

In [3]:
START_DATE = "2026-01-01"
END_DATE   = "2026-12-31"
API_KEY    = "rl_CCb2DLsiN6h5dW3hdtkUdv2fc" # Replace API key with valid stack exchange API which will allow 10,000 requests per day
OUTPUT_CSV_PATH    = "dataset/questions_2026.csv"
API = "https://api.stackexchange.com/2.3"

## Helper functions

In [4]:
def clean_question_body(html):
    if not html:
        return "", ""
    soup = BeautifulSoup(html, "lxml")
    pres = soup.find_all("pre")
    code = "\n\n".join(pre.get_text("\n", strip=True) for pre in pres)
    for pre in pres:
        pre.decompose()
    text = " ".join(soup.get_text(" ", strip=True).split())
    return text, code

def load_seen_question_ids(file_path):
    if not os.path.exists(file_path):
        return set()
    with open(file_path, newline="", encoding="utf-8") as file:
        return {row["question_id"] for row in csv.DictReader(file)}

def append_rows(path, rows):
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as file:
        w = csv.DictWriter(file, fieldnames=[
            "question_id",
            "title",
            "body_text",
            "code_text",
            "tags",
            "score",
            "view_count",
            "answer_count",
            "is_answered",
            "has_accepted_answer",
            "creation_date",
            "closed_reason",
            "duplicate_of"
        ])
        if new_file:
            w.writeheader()
        w.writerows(rows)

def get_filter_for_questions(api_key):
    params = {
        "include": "question.body;question.closed_details;question.closed_reason",
        "base": "default",
        "unsafe": "false"
    }
    if api_key:
        params["key"] = api_key
    response = requests.post(f"{API}/filters/create", data=params, timeout=30)
    try:
        data = response.json()
    except ValueError:
        raise RuntimeError("Quota exceeded")

    return data["items"][0]["filter"]

def parse_question(question):
    body, code = clean_question_body(question.get("body", ""))
    originals = (question.get("closed_details") or {}).get("original_questions") or []

    return {
        "question_id": question["question_id"],
        "title": BeautifulSoup(question.get("title", ""), "lxml").get_text(),
        "body_text": body,
        "code_text": code,
        "tags": "|".join(question.get("tags", [])),
        "score": question.get("score", 0),
        "view_count": question.get("view_count", 0),
        "answer_count": question.get("answer_count", 0),
        "is_answered": question.get("is_answered", False),
        "has_accepted_answer": "accepted_answer_id" in question,
        "creation_date": datetime.fromtimestamp(question["creation_date"], timezone.utc).isoformat(),
        "closed_reason": question.get("closed_reason", ""),
        "duplicate_of": "|".join(str(original["question_id"]) for original in originals),
    }

## Function to collect data

In [5]:
def collect(start_date, end_date, api_key, output_file):
    filter_param = get_filter_for_questions(api_key)
    existing_rows = load_seen_question_ids(output_file)
    total = len(existing_rows)
    from_date = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    to_date = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)

    while from_date <= to_date:
        page, has_more = 1, True
        while has_more:
            params = {
                "site": "stackoverflow",
                "sort": "creation",
                "order": "asc",
                "pagesize": 100,
                "page": page,
                "filter": filter_param,
                "fromdate": int(from_date.timestamp()),
                "todate": int((from_date + timedelta(days=1)).timestamp()) - 1
            }
            if api_key:
                params["key"] = api_key

            try:
                response = requests.get(f"{API}/questions", params=params, timeout=30)
                data = response.json()
            except (requests.RequestException, ValueError) as e:
                print("Request failed, something went wrong", e)
                return

            rows = []
            for question in data.get("items", []):
                question_id = str(question["question_id"])
                if question_id not in existing_rows:
                    rows.append(parse_question(question))
                    existing_rows.add(question_id)

            append_rows(output_file, rows)
            total += len(rows)
            print(f"{from_date.date()} page: {page}: +{len(rows)} rows added (total: {total}, quota left: {data.get('quota_remaining')})")

            if data.get("quota_remaining", 1) == 0:
                print("Daily quota used")
                return

            has_more = data.get("has_more", False)
            page += 1
            time.sleep(0.1)
        from_date += timedelta(days=1)

## Collect the data

In [6]:
collect(START_DATE, END_DATE, API_KEY, OUTPUT_CSV_PATH)

2026-01-01 page: 1: +0 rows added (total: 20142, quota left: 9696)
2026-01-02 page: 1: +0 rows added (total: 20142, quota left: 9696)
2026-01-02 page: 2: +0 rows added (total: 20142, quota left: 9694)
2026-01-03 page: 1: +0 rows added (total: 20142, quota left: 9694)
2026-01-04 page: 1: +0 rows added (total: 20142, quota left: 9692)
2026-01-05 page: 1: +0 rows added (total: 20142, quota left: 9692)
2026-01-05 page: 2: +0 rows added (total: 20142, quota left: 9690)
2026-01-06 page: 1: +0 rows added (total: 20142, quota left: 9690)
2026-01-06 page: 2: +0 rows added (total: 20142, quota left: 9688)
2026-01-07 page: 1: +0 rows added (total: 20142, quota left: 9688)
2026-01-07 page: 2: +0 rows added (total: 20142, quota left: 9686)
2026-01-08 page: 1: +0 rows added (total: 20142, quota left: 9686)
2026-01-08 page: 2: +0 rows added (total: 20142, quota left: 9684)
2026-01-09 page: 1: +0 rows added (total: 20142, quota left: 9684)
2026-01-09 page: 2: +0 rows added (total: 20142, quota left: 9

## Verify the data collection

In [7]:
df = pd.read_csv(OUTPUT_CSV_PATH)
print("No. of questions: ", len(df))
df.head()

No. of questions:  20225


,question_id,title,body_text,code_text,tags,score,view_count,answer_count,is_answered,has_accepted_answer,creation_date,closed_reason,duplicate_of
0,79858431,MUI DatePicker border color won’t change on fo...,I’m trying to change the border color of a MUI...,import { AdapterDayjs } from '@mui/x-date-pick...,material-ui|mui-x-date-picker,1,138,1,False,False,2026-01-01T00:09:35+00:00,NaN,NaN
1,79858435,PHP Variable Naming,If I have several variables that are only diff...,<?php \n $_tbl = 2; \n $tbl_Fld_type ...,php|variable-variables,-2,5975,7,False,False,2026-01-01T00:32:49+00:00,Duplicate,4656623|11740231|16955258|40549069|56754615
2,79858472,Spotfire: Rename columns in data tables - fool...,I had developed a dashboard with many pages an...,NaN,spotfire|spotfire-analyst|best-practices,0,82,0,False,False,2026-01-01T05:21:39+00:00,NaN,NaN
3,79858478,How can I use an OR combination in the evidenc...,When querying a BN model developed using py-bn...,"evidence = {'Y': 1, 'Y': 2} # but this just se...",py-bnlearn,0,23,0,False,False,2026-01-01T05:52:36+00:00,NaN,NaN
4,79858481,ViewControllerRegistry register view with model,Is there a way to bind data when registering a...,public void addViewControllers(ViewControllerR...,spring|spring-mvc|spring-webflux|best-practices,0,56,2,False,False,2026-01-01T05:59:29+00:00,NaN,NaN


## References

- https://link.sc/blog/how-to-scrape-stack-overflow
- https://api.stackexchange.com/docs/filters